# 🟠 Colab E-ii — TF Built-in Layers + GradientTape
**tf.keras.layers.Dense + manual training loop**

In [ ]:
import tensorflow as tf, numpy as np, matplotlib.pyplot as plt
tf.random.set_seed(42); np.random.seed(42)
print('TF:', tf.__version__)

## 📊 Section 1 — Data

In [ ]:
N=1000
x1=np.random.uniform(-2,2,(N,1)); x2=np.random.uniform(-2,2,(N,1)); x3=np.random.uniform(-2,2,(N,1))
y=2*x1**2+3*x2*x3+np.sin(x1*x2)+0.5*x3**2+np.random.normal(0,0.1,(N,1))
X=np.hstack([x1,x2,x3])
X_n=(X-X.mean(0))/X.std(0); y_mean,y_std=float(y.mean()),float(y.std()); y_n=(y-y_mean)/y_std
sp=int(0.8*N)
Xtr=tf.constant(X_n[:sp],dtype=tf.float32); Xte=tf.constant(X_n[sp:],dtype=tf.float32)
Ytr=tf.constant(y_n[:sp],dtype=tf.float32); Yte=tf.constant(y_n[sp:],dtype=tf.float32)

## 🏗️ Section 2 — Layer Objects (no Model wrapping yet)

In [ ]:
Dense=tf.keras.layers.Dense
l1=Dense(64,activation='relu',kernel_initializer='he_normal')
l2=Dense(32,activation='relu',kernel_initializer='he_normal')
l3=Dense(16,activation='tanh')
lo=Dense(1)

def forward(X):
    return lo(l3(l2(l1(X))))

# Trigger build by running once
_ = forward(Xtr[:1])
all_vars = l1.trainable_variables+l2.trainable_variables+l3.trainable_variables+lo.trainable_variables
print(f'Layers built. Total params: {sum(tf.size(v).numpy() for v in all_vars)}')

## 🏋️ Section 3 — GradientTape Training

In [ ]:
optimizer=tf.keras.optimizers.Adam(1e-3)
EPOCHS,BATCH=1000,64; history=[]
ds=(tf.data.Dataset.from_tensor_slices((Xtr,Ytr)).shuffle(800).batch(BATCH))

for epoch in range(EPOCHS):
    for Xb,Yb in ds:
        with tf.GradientTape() as tape:
            pred=forward(Xb)
            loss=tf.reduce_mean((pred-Yb)**2)
        grads=tape.gradient(loss,all_vars)
        optimizer.apply_gradients(zip(grads,all_vars))
    history.append(float(loss))
    if (epoch+1)%100==0:
        vl=float(tf.reduce_mean((forward(Xte)-Yte)**2))
        print(f'Ep {epoch+1:4d} | Loss: {float(loss):.5f} | Val: {vl:.5f}')

## 📈 Section 4 — Results

In [ ]:
plt.figure(figsize=(10,4))
plt.semilogy(history,'orange',lw=1.5); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('Colab E-ii — TF Built-in + GradientTape'); plt.grid(alpha=0.3); plt.show()

yp=forward(Xte).numpy()*y_std+y_mean; yt=Yte.numpy()*y_std+y_mean
plt.figure(figsize=(6,6))
plt.scatter(yt,yp,alpha=0.4,s=15,color='orange')
mn,mx=yt.min(),yt.max(); plt.plot([mn,mx],[mn,mx],'r--',lw=2)
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title('Colab E-ii — Predicted vs Actual'); plt.grid(alpha=0.3); plt.show()
print(f'Test MSE: {((yp-yt)**2).mean():.4f}')